In [ ]:
# ВНИМАНИЕ: парсер разработан для работы на территории Китая, где сервисы Google (в том числе официальные источники ChromeDriver) заблокированы.
# Поэтому для запуска необходимо вручную скачать chromedriver.exe с китайского зеркала https://chromedrive.cn/,
# выбрав версию, соответствующую (или максимально близкую) вашей версии браузера Google Chrome.
# Скачанный файл положите в ту же папку, что и ноутбук, в коде прописан прямой путь Service("chromedriver.exe").

# парсер собирает заголовок, ссылку, источник (主站新闻 или 地方新闻), дату публикации статьи, текст статьи
# на сайте renmin ribao в поиске максимум отдает ~334 страницы
# если на страничке картинка, то текст может собираться некорректно/не собираться -- требуется доработка
# ИСПРАВЛЕНИЯ:
# 1) включён реальный фильтр по min_date / max_date
# 2) добавлена локальная сортировка итоговой таблицы по дате
# 3) добавлена дедупликация не только по URL, но и по заголовку+дню
# 4) уменьшен шум поиска: isFuzzy=False
# 5) добавлены комментарии "# ДОБАВЛЕНО" и "# ИЗМЕНЕНО" у новых/изменённых строк

In [ ]:
import time
import json
import re  # ДОБАВЛЕНО: понадобится для нормализации заголовков
import requests
import pandas as pd
import traceback
from datetime import datetime
from urllib.parse import quote, urlsplit, urlunsplit  # ДОБАВЛЕНО: нормализация URL
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

keyword = "出口"  # ключевое слово
quoted_keyword = quote(keyword)

# Часть 1: Получаем cookie через Selenium
chrome_options = Options()

# chrome_options.add_argument("--headless")  # можно включить, если всё работает
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

service = Service("chromedriver.exe")   # Примечание: chromedriver.exe скачан вручную с китайского зеркала chromedrive.cn из-за блокировки официальных серверов Google.
driver = webdriver.Chrome(service=service, options=chrome_options)
wait = WebDriverWait(driver, 10)

start_url = f"http://search.people.cn/s?keyword={keyword}&st=1"
print("Открываем браузер для получения cookies...")
driver.get(start_url)
time.sleep(3)
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(2)

cookies_list = driver.get_cookies()

# Сохраняем рабочий браузер для сбора полных текстов
article_driver = driver

# Формируем cookie-строку
cookies = "; ".join([f"{c['name']}={c['value']}" for c in cookies_list])

# Часть 2: Запрос через requests с актуальными cookie
url = "http://search.people.cn/search-platform/front/search"
headers = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Encoding": "gzip, deflate",
    "Accept-Language": "ru,en-GB;q=0.9,en-US;q=0.8,en;q=0.7,zh-CN;q=0.6,zh;q=0.5",
    "Connection": "keep-alive",
    "Content-Type": "application/json;charset=UTF-8",
    "DNT": "1",
    "Host": "search.people.cn",
    "Origin": "http://search.people.cn",
    "Referer": f"http://search.people.cn/s/?keyword={quoted_keyword}&st=1",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36",
    "Cookie": cookies
}

articles = []
max_pages = 50  # выставить нужное количество страниц

# Ограничение по дате
min_date = datetime(2024, 1, 1)
max_date = datetime.now()

source_type_map = {
    1: "主站新闻",
    2: "地方新闻"
}

stop_phrases = [
    "责编：", "责任编辑：", "人民网", "人民日报", "版权所有",
    "京ICP", "京公网安备", "联系我们", "数据服务", "违法和不良信息", "jubao@", "rmwjubao@",
    "copyright", "Copyright", "举报电话", "声明", "服务许可证", "招聘", "供稿", "广告服务"
]

# ДОБАВЛЕНО: функция для нормализации URL перед дедупликацией
def normalize_url(raw_url: str) -> str:
    raw_url = str(raw_url or "").strip()
    if not raw_url:
        return ""
    parts = urlsplit(raw_url)
    # Убираем query и fragment, чтобы одинаковые статьи с хвостами не считались разными
    return urlunsplit((parts.scheme, parts.netloc, parts.path, "", ""))

# ДОБАВЛЕНО: функция для нормализации заголовка
def normalize_title(raw_title: str) -> str:
    raw_title = str(raw_title or "").strip()
    raw_title = BeautifulSoup(raw_title, "html.parser").get_text()
    raw_title = re.sub(r"\s+", "", raw_title)
    return raw_title

# ДОБАВЛЕНО: более аккуратное извлечение текста статьи
def extract_article_text(link: str) -> str:
    try:
        article_driver.get(link)
        time.sleep(2)

        # ДОБАВЛЕНО: сначала пытаемся найти более вероятные контейнеры основного текста
        candidate_selectors = [
            "div.rm_txt_con",
            "div#rm_txt_con",
            "div.box_con",
            "div.content",
            "div.article",
            "div.layout.rm_txt_con"
        ]

        paragraphs = []
        for selector in candidate_selectors:
            elems = article_driver.find_elements(By.CSS_SELECTOR, f"{selector} p")
            if elems:
                paragraphs = elems
                break

        # ДОБАВЛЕНО: fallback — если контейнеры не сработали, берём все <p>
        if not paragraphs:
            wait.until(EC.presence_of_all_elements_located((By.TAG_NAME, "p")))
            paragraphs = article_driver.find_elements(By.TAG_NAME, "p")

        full_text_parts = []
        for p in paragraphs:
            text = p.text.strip()
            if not text:
                continue
            if any(phrase in text for phrase in stop_phrases):
                break
            full_text_parts.append(text)

        full_text = " ".join(full_text_parts).strip()
        return full_text

    except Exception:
        print("Не удалось загрузить статью:", link)
        return ""

pages_with_hits = 0  # ДОБАВЛЕНО: счётчик страниц, где нашлись статьи в нужном диапазоне
skipped_by_date = 0  # ДОБАВЛЕНО: сколько результатов отброшено по дате

for page in range(1, max_pages + 1):
    print("Сканирую страницу", page)

    payload = {
        "key": keyword,
        "page": page,
        "limit": 10,
        "hasTitle": True,
        "hasContent": True,
        "isFuzzy": False,  # ИЗМЕНЕНО: уменьшаем шум и лишние расширенные совпадения
        "type": 1,
        "sortType": 2,     # оставляем как у вас; итоговую строгую сортировку делаем локально ниже
        "startTime": 0,    # ИЗМЕНЕНО: серверный фильтр не используем как единственный источник правды
        "endTime": 0
    }

    try:
        for attempt in range(3):
            try:
                response = requests.post(url, json=payload, headers=headers, timeout=10)
                break
            except requests.exceptions.ConnectionError:
                print(f"Попытка {attempt + 1} не удалась. Повтор через 5 сек...")
                time.sleep(5)
        else:
            print("Не удалось установить соединение после 3 попыток.")
            continue

        response.encoding = "utf-8"

        if response.status_code != 200:
            print("Код ответа:", response.status_code)
            print("Содержимое (обрезано):", response.text[:200])
            break

        data = response.json()
        results = data.get("data", {}).get("records", [])

        if not results:
            print("Нет результатов. Останавливаемся.")
            print("\nОтвет сервера:")
            print(response.text[:500])
            break

        added_this_page = 0  # ДОБАВЛЕНО: сколько статей реально добавлено с этой страницы

        for item in results:
            raw_title = str(item.get("title", "")).strip()
            title = BeautifulSoup(raw_title, "html.parser").get_text()  # удаляем <em>
            link = str(item.get("url", "")).strip()
            source = source_type_map.get(item.get("source"), str(item.get("source")))

            # ДОБАВЛЕНО: сначала приводим дату к datetime, только потом сравниваем с min_date / max_date
            display_time = item.get("displayTime", 0)
            if not display_time:
                skipped_by_date += 1
                continue

            article_dt = datetime.fromtimestamp(display_time / 1000)

            # ДОБАВЛЕНО: реальный фильтр по диапазону дат
            if article_dt < min_date or article_dt > max_date:
                skipped_by_date += 1
                continue

            date = article_dt.strftime("%Y-%m-%d %H:%M")

            full_text = extract_article_text(link)

            articles.append({
                "Заголовок": title,
                "Ссылка": link,
                "Источник": source,
                "Дата": date,
                "Текст": full_text
            })
            added_this_page += 1

        if added_this_page > 0:
            pages_with_hits += 1

        print(
            f"Сканирую страницу {page} | Получено результатов: {len(results)} | "
            f"Добавлено в диапазоне дат: {added_this_page} | Всего собрано: {len(articles)}"
        )
        time.sleep(3)

    except Exception:
        print("Ошибка на странице", page, ":")
        traceback.print_exc()
        break

article_driver.quit()

df = pd.DataFrame(articles)

# ДОБАВЛЕНО: защитный блок, если после фильтра ничего не собралось
if df.empty:
    print("\nПосле фильтрации по дате не осталось статей. Проверьте keyword / max_pages / min_date.")
else:
    # ДОБАВЛЕНО: приводим дату к datetime для сортировки и дедупликации
    df["Дата"] = pd.to_datetime(df["Дата"], errors="coerce")

    # ДОБАВЛЕНО: нормализуем URL и заголовок для более аккуратной дедупликации
    df["Ссылка_norm"] = df["Ссылка"].map(normalize_url)
    df["Заголовок_norm"] = df["Заголовок"].map(normalize_title)
    df["Дата_день"] = df["Дата"].dt.strftime("%Y-%m-%d")

    before = len(df)

    # ДОБАВЛЕНО: сначала удаляем дубли по нормализованной ссылке
    df = df.drop_duplicates(subset=["Ссылка_norm"], keep="first")

    # ДОБАВЛЕНО: затем удаляем повторы одного и того же заголовка в один и тот же день
    df = df.drop_duplicates(subset=["Заголовок_norm", "Дата_день"], keep="first")

    # ДОБАВЛЕНО: можно убрать пустые даты на всякий случай
    df = df.dropna(subset=["Дата"])

    # ДОБАВЛЕНО: строгая локальная сортировка по дате
    df = df.sort_values("Дата", ascending=False).reset_index(drop=True)

    # ДОБАВЛЕНО: сохраняем дату в человекочитаемом виде
    df["Дата"] = df["Дата"].dt.strftime("%Y-%m-%d %H:%M")

    removed = before - len(df)

    # ДОБАВЛЕНО: удаляем технические колонки перед сохранением
    df = df.drop(columns=["Ссылка_norm", "Заголовок_norm", "Дата_день"])

    output_file = "people_cn_出口.xlsx"  
    df.to_excel(output_file, index=False)

    print(f"\nСохранено {len(df)} статей в {output_file}")
    print(f"Отброшено по дате: {skipped_by_date}")
    print(f"Удалено дублей после сборки: {removed}")
    print(f"Страниц, где нашлись статьи в диапазоне дат: {pages_with_hits}")
